In [1]:
import pandas as pd

In [15]:
df = pd.read_csv(r"C:\Users\geral\Documents\Memoria\experiments\benchmark_final\benchmark_final.csv")

In [22]:
df.groupby("Modelo").count()

,test_mse,test_rmse,test_mae,test_mape,test_loss,val_mse,val_rmse,val_mae,val_mape,val_loss,...,n_train,n_val,n_test,test_ar_mape,train_num_targets,eval_num_targets,ft_mode,ft_lr,ft_unfreeze_lr,ft_freeze_encoder_epochs
Modelo,,,,,,,,,,,,,,,,,,,,,
CoFormer-Uni,15,15,15,15,15,15,15,15,15,15,...,15,15,15,0,15,15,0,0,0,0
Custom,17,17,17,17,17,17,17,17,17,17,...,17,17,17,0,6,6,0,0,0,0
Custom-AttnPool,16,16,16,16,16,16,16,16,16,16,...,16,16,16,0,15,15,0,0,0,0
Custom-AttnPool-Small,16,16,16,16,16,16,16,16,16,16,...,16,16,16,0,15,15,0,0,0,0
Custom-Small,17,17,17,17,17,17,17,17,17,17,...,17,17,17,0,6,6,0,0,0,0
Custom-T2V,1,1,1,1,1,1,1,1,1,1,...,1,1,1,0,0,0,0,0,0,0
Custom-T2V-Small,2,2,2,2,2,2,2,2,2,2,...,2,2,2,0,0,0,0,0,0,0
Custom-TempBias,15,15,15,15,15,15,15,15,15,15,...,15,15,15,0,15,15,0,0,0,0
Custom-TempBias-Small,15,15,15,15,15,15,15,15,15,15,...,15,15,15,0,15,15,0,0,0,0


In [14]:
df.iloc[90:].groupby("Modelo")["train_time_s"].mean()

Modelo
Custom              2164.302500
Custom-Small        2743.875714
Custom-T2V          6239.300000
Custom-T2V-Small    4511.875000
Linear              1398.803333
NoTargetToken       3261.147500
NoTimeEncoding      2268.310000
Persistence            0.000000
STraTS_Adapter      2865.710909
Name: train_time_s, dtype: float64

In [1]:
!pip install colorama

In [2]:
from google.colab import drive
import sys
import os

# Montar Drive
drive.mount('/content/drive')

# Raiz del proyecto en Drive
PROJECT_ROOT = "/content/drive/MyDrive/Memoria"

# Forzar cwd para que rutas relativas (experiments/, configs/, data/) funcionen
if os.path.isdir(PROJECT_ROOT):
    os.chdir(PROJECT_ROOT)
else:
    raise FileNotFoundError(f"No existe PROJECT_ROOT: {PROJECT_ROOT}")

# Asegurar imports del proyecto
src_path = os.path.join(PROJECT_ROOT, "src")
scripts_path = os.path.join(PROJECT_ROOT, "scripts")
for p in (src_path, scripts_path):
    if p not in sys.path:
        sys.path.append(p)

print("Working directory:", os.getcwd())
print("src in sys.path:", src_path in sys.path)
print("scripts in sys.path:", scripts_path in sys.path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Memoria
src in sys.path: True
scripts in sys.path: True


In [3]:
"""
benchmark_final.py — Benchmark integral para la tesis.

Ejecuta de forma resumible:
  1. Todos los 17 datasets reales.
  2. Múltiples semillas por dataset (default: 3).
  3. Modelos: Custom, STraTS_Adapter, Persistence, Linear,
              NoTimeEncoding (ablación), NoTargetToken (ablación).
              (CoFormer está comentado, listo para activar)
  4. Evaluación en SPLIT DE TEST real (separado de validación).
  5. Medición de costo (tiempo de entrenamiento, nº de parámetros).
  6. Análisis estadístico al final (Wilcoxon, bootstrap, tablas resumen).

Uso:
  python scripts/benchmark_final.py
  python scripts/benchmark_final.py --start-dataset 7 --end-dataset 17
  python scripts/benchmark_final.py --seeds 42 84 126
  python scripts/benchmark_final.py --skip-ablation   (sólo modelos principales)

Se puede detener con Ctrl+C y reanudar ejecutando el mismo comando.
El progreso se guarda en el CSV después de cada modelo entrenado.
"""

import os
import sys
import argparse
import copy
import time
from colorama import init, Fore, Style
import pandas as pd
import numpy as np

# __file__ = os.path.abspath(r"C:\Users\geral\Documents\Memoria\scripts\benchmark_final.py")

# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
sys.path.append(os.path.abspath("/content/drive/MyDrive/Memoria/src"))

import torch
from torch.utils.data import DataLoader

from ts_transformer.data import (
    TimeSeriesDataset, EventTimeSeriesDataset, SequenceBuilder,
    build_collate_fn, StandardScaler, split_dataframe_by_time
)
from ts_transformer.data.timeseries_dataset import TimeSeriesDatasetConfig
from ts_transformer.models import TimeSeriesTransformer, TimeSeriesEncoderDecoder
from ts_transformer.training import TrainingConfig, Trainer
from ts_transformer.utils import (
    load_data_config, load_model_config, load_training_config,
    set_global_seed, setup_logging, get_logger
)

from state_art.strats.model import STraTSNetwork
# from state_art.coformer.model import CompatibleTransformer
from state_art.baselines_wrapper import MultiHorizonBaselineWrapper
from state_art.simple_baselines import (
    PersistenceModel, LinearBaselineModel,
    NoTimeEncodingTransformer, NoTargetTokenTransformer,
)

from ar_finetuning import run_ar_finetuning

In [4]:
# ======================================================================
# Argumentos CLI
# ======================================================================
def parse_args():
    p = argparse.ArgumentParser(
        description="Benchmark Final — Tesis Memoria",
        formatter_class=argparse.RawTextHelpFormatter,
    )
    p.add_argument("--base-data-config", type=str,
                   default="configs/data/real_data1.yaml")
    p.add_argument("--model-config", type=str,
                   default="configs/model/transformer_base_real_data1.yaml")
    p.add_argument("--training-config", type=str,
                   default="configs/training/default_real_data1.yaml")
    p.add_argument("--start-dataset", type=int, default=1)
    p.add_argument("--end-dataset", type=int, default=17)
    p.add_argument("--seeds", type=int, nargs="+", default=[42, 84, 126],
                   help="Lista de semillas aleatorias (default: 42 84 126)")
    p.add_argument("--skip-ablation", action="store_true",
                   help="Si se activa, sólo entrena Custom, STraTS, Persistence, Linear")
    p.add_argument("--skip-baselines", action="store_true",
                   help="Si se activa, sólo entrena Custom y STraTS")
    p.add_argument("--reverse-datasets", action="store_true",
                   help="Ejecuta datasets en orden inverso (fin -> inicio)")
    p.add_argument("--exp-dir", type=str, default="/content/drive/MyDrive/Memoria/experiments/benchmark_final",
                   help="Directorio de salida")
    # En notebook (ipykernel), ignorar argumentos inyectados por Jupyter (ej. -f ...)
    if "ipykernel" in sys.modules:
        args, _ = p.parse_known_args()
        return args
    return p.parse_args()


# ======================================================================
# Utilidades
# ======================================================================
def _timestamps_to_float(col: pd.Series) -> np.ndarray:
    if np.issubdtype(col.dtype, np.datetime64):
        return (col.view("int64") / 1e9).astype("float32")
    return col.astype("float32").to_numpy()


def count_parameters(model: torch.nn.Module) -> int:
    """Cuenta parámetros entrenables."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_parameters(model: torch.nn.Module) -> int:
    """Cuenta parámetros totales."""
    return sum(p.numel() for p in model.parameters())


# ======================================================================
# Construcción de modelos
# ======================================================================
def build_models(
    model_cfg,
    use_events: bool,
    model_input_dim: int,
    input_dim: int,
    output_dim: int,
    skip_ablation: bool = False,
    skip_baselines: bool = False,
):
    """
    Construye todos los modelos a evaluar.

    Returns
    -------
    dict[str, nn.Module]
        Nombre → modelo instanciado.
    dict[str, bool]
        Nombre → True si el modelo requiere entrenamiento.
    """
    model_cfg.input_dim = model_input_dim
    model_cfg.output_dim = output_dim
    model_cfg.use_sensor_embedding = bool(use_events)
    model_cfg.num_sensors = input_dim if use_events else 0

    d_model = model_cfg.d_model

    models = {}
    trainable = {}

    cfg_small = load_model_config("configs/model/transformer_small.yaml")
    cfg_small.input_dim = model_input_dim
    cfg_small.output_dim = output_dim
    cfg_small.use_sensor_embedding = bool(use_events)
    cfg_small.num_sensors = input_dim if use_events else 0

    cfg_large = load_model_config("configs/model/transformer_large.yaml")
    cfg_large.input_dim = model_input_dim
    cfg_large.output_dim = output_dim
    cfg_large.use_sensor_embedding = bool(use_events)
    cfg_large.num_sensors = input_dim if use_events else 0

    # --- Modelo propuesto ---
    models["Custom-Small"] = TimeSeriesTransformer(copy.deepcopy(cfg_small))
    trainable["Custom-Small"] = True
    models["Custom"] = TimeSeriesTransformer(copy.deepcopy(model_cfg))
    trainable["Custom"] = True
    models["Custom-Large"] = TimeSeriesTransformer(copy.deepcopy(cfg_large))
    trainable["Custom-Large"] = True

    # --- Variante Encoder-Decoder ---
    models["EncDec-Small"] = TimeSeriesEncoderDecoder(copy.deepcopy(cfg_small))
    trainable["EncDec-Small"] = True
    models["EncDec"] = TimeSeriesEncoderDecoder(copy.deepcopy(model_cfg))
    trainable["EncDec"] = True
    models["EncDec-Large"] = TimeSeriesEncoderDecoder(copy.deepcopy(cfg_large))
    trainable["EncDec-Large"] = True

    # --- STraTS Adapter ---
    # num_features = input_dim + 1: el +1 es para el feature_id marcador
    # de target (= D) que permite a STraTS distinguir tripletas de historia
    # de tripletas target en el transformer (comparación justa).
    s_base = STraTSNetwork(
        num_features=input_dim + 1,
        d_model=d_model,
        num_classes=output_dim,
    )
    models["STraTS_Adapter"] = MultiHorizonBaselineWrapper(
        s_base, "strats", d_model, output_dim, use_sensor_embedding=use_events
    )
    trainable["STraTS_Adapter"] = True

    # --- CoFormer Adapter (comentado, listo para activar) ---
    # c_base = CompatibleTransformer(
    #     num_variates=input_dim if use_events else 1,
    #     d_model=d_model,
    #     num_classes=output_dim,
    # )
    # models["CoFormer_Adapter"] = MultiHorizonBaselineWrapper(
    #     c_base, "coformer", d_model, output_dim, use_sensor_embedding=use_events
    # )
    # trainable["CoFormer_Adapter"] = True

    if not skip_baselines:
        # --- Persistence (no entrena) ---
        models["Persistence"] = PersistenceModel(
            input_dim=model_input_dim, output_dim=output_dim
        )
        trainable["Persistence"] = False

        # --- Linear Baseline ---
        models["Linear"] = LinearBaselineModel(
            input_dim=model_input_dim,
            output_dim=output_dim,
            d_model=64,
            max_history=50,
        )
        trainable["Linear"] = True

    if not skip_ablation:
        # --- Ablación: sin encoding temporal continuo ---
        models["NoTimeEncoding"] = NoTimeEncodingTransformer(copy.deepcopy(model_cfg))
        trainable["NoTimeEncoding"] = True

        # --- Ablación: sin target token ---
        models["NoTargetToken"] = NoTargetTokenTransformer(copy.deepcopy(model_cfg))
        trainable["NoTargetToken"] = True

    return models, trainable


# ======================================================================
# Entrenamiento y evaluación de un modelo
# ======================================================================
def train_and_evaluate(
    model,
    model_name: str,
    needs_training: bool,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    training_cfg: TrainingConfig,
    checkpoint_dir: str,
    device: torch.device,
    logger,
) -> dict:
    """
    Entrena un modelo (si necesita entrenamiento), lo evalúa en test,
    y retorna un diccionario con métricas + costo.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    cfg_mod = copy.deepcopy(training_cfg)
    cfg_mod.checkpoint_dir = checkpoint_dir

    n_params_trainable = count_parameters(model)
    n_params_total = count_total_parameters(model)

    if needs_training:
        # Entrenar
        logger.info(f"    Entrenando {model_name} ({n_params_trainable:,} params)...")
        trainer = Trainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=cfg_mod,
        )

        t_start = time.time()
        history = trainer.fit()
        train_time = time.time() - t_start
        epochs_run = len(history.get("train_loss", []))

        # Cargar mejor checkpoint
        ckpt_path = os.path.join(checkpoint_dir, "best_model.pt")
        if os.path.exists(ckpt_path):
            checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
            if "model_state_dict" in checkpoint:
                model.load_state_dict(checkpoint["model_state_dict"])
            else:
                model.load_state_dict(checkpoint)
    else:
        # Modelo sin entrenamiento (e.g. Persistence)
        model = model.to(device)
        trainer = Trainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=cfg_mod,
        )
        train_time = 0.0
        epochs_run = 0

    # Evaluación en TEST (split real separado)
    logger.info(f"    Evaluando {model_name} en TEST...")
    test_metrics = trainer.evaluate_on_loader(test_loader, prefix="test_")

    # Evaluación en VAL para comparación
    val_metrics = trainer.evaluate_on_loader(val_loader, prefix="val_")

    record = {}
    record.update(test_metrics)
    record.update(val_metrics)
    record["train_time_s"] = round(train_time, 2)
    record["epochs_run"] = epochs_run
    record["n_params_trainable"] = n_params_trainable
    record["n_params_total"] = n_params_total

    return record


# ======================================================================
# Pipeline de datos
# ======================================================================
def prepare_data(base_data_cfg, ds_idx, logger):
    """Carga y prepara un dataset concreto. Retorna loaders y metadatos."""
    target_csv = f"data/processed/real_data_{ds_idx}.csv"
    if not os.path.exists(target_csv):
        logger.warning(
            Fore.YELLOW
            + f"Saltando Dataset {ds_idx}: no existe {target_csv}"
            + Style.RESET_ALL
        )
        return None

    df = pd.read_csv(target_csv)
    time_col = base_data_cfg.time_column

    # Asegurar timestamps numéricos
    if not pd.api.types.is_numeric_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col]).apply(lambda x: x.timestamp())

    df = df.sort_values(time_col).reset_index(drop=True)
    df_train, df_val, df_test = split_dataframe_by_time(
        df, time_column=time_col,
        train_ratio=base_data_cfg.train_ratio,
        val_ratio=base_data_cfg.val_ratio,
    )

    def process_split(df_part):
        ts = _timestamps_to_float(df_part[time_col])
        X = df_part[base_data_cfg.feature_columns].to_numpy(dtype="float32")
        y = df_part[base_data_cfg.target_columns].to_numpy(dtype="float32")
        return ts, X, y

    ts_train, X_train, y_train = process_split(df_train)
    ts_val, X_val, y_val = process_split(df_val)
    ts_test, X_test, y_test = process_split(df_test)

    v_scal, t_scal = StandardScaler(), StandardScaler()
    X_train_s = v_scal.fit_transform(X_train)
    y_train_s = t_scal.fit_transform(y_train)
    X_val_s = v_scal.transform(X_val)
    y_val_s = t_scal.transform(y_val)
    X_test_s = v_scal.transform(X_test)
    y_test_s = t_scal.transform(y_test)

    v_tr = np.concatenate([X_train_s, y_train_s], axis=1)
    v_va = np.concatenate([X_val_s, y_val_s], axis=1)
    v_te = np.concatenate([X_test_s, y_test_s], axis=1)
    input_dim = X_train.shape[1]
    output_dim = y_train.shape[1]

    ds_cfg = TimeSeriesDatasetConfig(
        history_length=base_data_cfg.history_length,
        target_offset_choices=base_data_cfg.target_offset_choices,
        target_offset_min=base_data_cfg.target_offset_min,
        target_offset_max=base_data_cfg.target_offset_max,
        stride=base_data_cfg.stride,
        min_history_length=base_data_cfg.min_history_length,
        num_targets=base_data_cfg.num_targets
        if hasattr(base_data_cfg, "num_targets")
        else 3,
    )

    use_events = base_data_cfg.use_event_tokens

    if use_events:
        ds_tr = EventTimeSeriesDataset(
            X_train_s, ts_train, y_train_s, ds_cfg, input_dim, output_dim
        )
        ds_va = EventTimeSeriesDataset(
            X_val_s, ts_val, y_val_s, ds_cfg, input_dim, output_dim
        )
        ds_te = EventTimeSeriesDataset(
            X_test_s, ts_test, y_test_s, ds_cfg, input_dim, output_dim
        )
        tsid = [
            (
                base_data_cfg.feature_columns.index(c)
                if c in base_data_cfg.feature_columns
                else input_dim
            )
            for c in base_data_cfg.target_columns
        ]
        sqb = SequenceBuilder(
            input_dim=1,
            target_token_value="zeros",
            use_sensor_ids=True,
            num_sensors=input_dim,
            num_target_tokens=output_dim,
            target_sensor_ids=tsid,
        )
        mi_dim = 1
    else:
        ds_tr = TimeSeriesDataset(v_tr, ts_train, ds_cfg, input_dim, output_dim)
        ds_va = TimeSeriesDataset(v_va, ts_val, ds_cfg, input_dim, output_dim)
        ds_te = TimeSeriesDataset(v_te, ts_test, ds_cfg, input_dim, output_dim)
        sqb = SequenceBuilder(
            input_dim=input_dim,
            target_token_value="zeros",
            use_sensor_ids=False,
            num_sensors=0,
            num_target_tokens=1,
        )
        mi_dim = input_dim

    collate_fn = build_collate_fn(sequence_builder=sqb, pad_to_max_length=True)

    num_workers = 0  # Seguro en Windows
    loader_kwargs = dict(
        batch_size=base_data_cfg.batch_size,
        collate_fn=collate_fn,
        pin_memory=True,
        num_workers=num_workers,
    )

    train_loader = DataLoader(ds_tr, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(ds_va, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(ds_te, shuffle=False, **loader_kwargs)

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "input_dim": input_dim,
        "output_dim": output_dim,
        "model_input_dim": mi_dim,
        "use_events": use_events,
        "n_train": len(ds_tr),
        "n_val": len(ds_va),
        "n_test": len(ds_te),
    }


# ======================================================================
# Main
# ======================================================================
def main():
    args = parse_args()
    init(autoreset=True)
    setup_logging()
    logger = get_logger("benchmark_final")

    exp_dir = args.exp_dir
    # Forzar que toda salida quede en Drive.
    if not os.path.isabs(exp_dir):
        exp_dir = os.path.join(PROJECT_ROOT, exp_dir)
    exp_dir_abs = os.path.abspath(exp_dir)
    project_root_abs = os.path.abspath(PROJECT_ROOT)
    if not exp_dir_abs.startswith(project_root_abs):
        logger.warning(
            Fore.YELLOW
            + f"exp_dir fuera de Drive detectado ({exp_dir_abs}). Redirigiendo a carpeta persistente."
            + Style.RESET_ALL
        )
        exp_dir_abs = os.path.join(project_root_abs, "experiments", "benchmark_final")
    exp_dir = exp_dir_abs
    os.makedirs(exp_dir, exist_ok=True)
    out_csv = os.path.join(exp_dir, "benchmark_final.csv")

    logger.info(
        Fore.GREEN
        + f"=== BENCHMARK FINAL ==="
        + Style.RESET_ALL
    )
    logger.info(f"  Datasets: {args.start_dataset} a {args.end_dataset}")
    logger.info(f"  Semillas: {args.seeds}")
    logger.info(f"  Ablación: {'NO' if args.skip_ablation else 'SÍ'}")
    logger.info(f"  Baselines simples: {'NO' if args.skip_baselines else 'SÍ'}")
    logger.info(f"  Directorio: {exp_dir}")

    # Cargar configs base (resolucion robusta de rutas en Drive)
    def _resolve_cfg_path(p: str) -> str:
        cands = []
        if os.path.isabs(p):
            cands.append(p)
        else:
            cands.append(os.path.join(PROJECT_ROOT, p))
            cands.append(os.path.join(PROJECT_ROOT, "src", p))
            cands.append(os.path.abspath(p))
        for c in cands:
            if os.path.exists(c):
                return c
        raise FileNotFoundError(
            f"No se encontro config: {p}. Candidatas: {cands}. "
            "Verifica que exista en /content/drive/MyDrive/Memoria/configs"
        )

    data_cfg_path = _resolve_cfg_path(args.base_data_config)
    model_cfg_path = _resolve_cfg_path(args.model_config)
    train_cfg_path = _resolve_cfg_path(args.training_config)

    base_data_cfg = load_data_config(data_cfg_path)
    model_cfg_base = load_model_config(model_cfg_path)
    training_cfg, _ = load_training_config(train_cfg_path)
    device = torch.device(
        training_cfg.device if torch.cuda.is_available() else "cpu"
    )

    # Cargar resultados previos para resume
    master_results = []
    completed_runs = set()
    if os.path.exists(out_csv):
        df_exist = pd.read_csv(out_csv)
        master_results = df_exist.to_dict("records")
        for row in master_results:
            completed_runs.add((row["Dataset_ID"], row["Seed"], row["Modelo"]))
        logger.info(
            Fore.YELLOW
            + f"  [RESUME] {len(completed_runs)} corridas previas detectadas."
            + Style.RESET_ALL
        )

    # Iterar sobre datasets (soporta orden inverso)
    if args.reverse_datasets or args.start_dataset > args.end_dataset:
        dataset_range = range(args.start_dataset, args.end_dataset - 1, -1)
    else:
        dataset_range = range(args.start_dataset, args.end_dataset + 1)

    for ds_idx in dataset_range:
        logger.info(
            Fore.CYAN
            + f"\n{'='*60}"
            + f"\n  DATASET {ds_idx}"
            + f"\n{'='*60}"
            + Style.RESET_ALL
        )

        data = prepare_data(base_data_cfg, ds_idx, logger)
        if data is None:
            continue

        logger.info(
            f"  Muestras: train={data['n_train']}, "
            f"val={data['n_val']}, test={data['n_test']}"
        )

        for seed in args.seeds:
            logger.info(
                Fore.MAGENTA
                + f"\n  [Dataset {ds_idx} | Seed {seed}]"
                + Style.RESET_ALL
            )
            set_global_seed(seed, deterministic=False)

            # Construir modelos frescos para cada semilla
            models, trainable = build_models(
                copy.deepcopy(model_cfg_base),
                data["use_events"],
                data["model_input_dim"],
                data["input_dim"],
                data["output_dim"],
                skip_ablation=args.skip_ablation,
                skip_baselines=args.skip_baselines,
            )

            for name, model in models.items():
                requires_base_training = (ds_idx, seed, name) not in completed_runs
                
                # Check for EncDec fine-tunings completion
                missing_fts = []
                if "EncDec" in name:
                    for mode in ["Contiguous", "Random", "Mixed"]:
                        if (ds_idx, seed, f"{name}_FT_AR_{mode}") not in completed_runs:
                            missing_fts.append(mode)

                if not requires_base_training and not missing_fts:
                    logger.info(f"    [SKIP] {name} y sus variantes (completados)")
                    continue
                    
                if not requires_base_training and "EncDec" in name and missing_fts:
                    logger.info(f"    [RESUME-FT] {name} base ya evaluado. Faltan fine-tunings: {missing_fts}")

                try:
                    ckpt_dir = os.path.join(
                        exp_dir, f"ds_{ds_idx}_seed_{seed}", name
                    )
                    if requires_base_training:
                        record = train_and_evaluate(
                            model=model,
                            model_name=name,
                            needs_training=trainable[name],
                            train_loader=data["train_loader"],
                            val_loader=data["val_loader"],
                            test_loader=data["test_loader"],
                            training_cfg=training_cfg,
                            checkpoint_dir=ckpt_dir,
                            device=device,
                            logger=logger,
                        )

                        record["Dataset_ID"] = ds_idx
                        record["Modelo"] = name
                        record["Seed"] = seed
                        record["n_train"] = data["n_train"]
                        record["n_val"] = data["n_val"]
                        record["n_test"] = data["n_test"]

                        master_results.append(record)

                        # Guardar progreso inmediatamente
                        pd.DataFrame(master_results).to_csv(out_csv, index=False)
                        logger.info(
                            Fore.GREEN
                            + f"    [OK] {name} ds={ds_idx} seed={seed} → "
                            + f"test_mse={record.get('test_mse', 'N/A'):.6f}, "
                            + f"time={record.get('train_time_s', 0):.1f}s"
                            + Style.RESET_ALL
                        )
                    else:
                        # Load the model's base weights for fine-tunings since base training was skipped
                        ckpt_path = os.path.join(ckpt_dir, "best_model.pt")
                        if os.path.exists(ckpt_path):
                            checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
                            model.load_state_dict(checkpoint.get("model_state_dict", checkpoint))
                            logger.info(Fore.BLUE + f"    [STATE] Cargados pesos base de {name} para reanudar FT." + Style.RESET_ALL)

                    # ------ INICIO FINE-TUNING RECURSIVO ------
                    if "EncDec" in name and missing_fts:
                        for mode in missing_fts:
                            ft_record = run_ar_finetuning(
                                mode=mode,
                                model=model,
                                model_name=name,
                                base_data_cfg=base_data_cfg,
                                ds_idx=ds_idx,
                                training_cfg=training_cfg,
                                base_ckpt_dir=os.path.join(exp_dir, f"ds_{ds_idx}_seed_{seed}"),
                                device=device,
                                logger=logger
                            )
                            if ft_record:
                                # Reempaquetar con el nombre nuevo reflejando el modo
                                ft_record["Dataset_ID"] = ds_idx
                                ft_record["Modelo"] = f"{name}_FT_AR_{mode}"
                                ft_record["Seed"] = seed
                                ft_record["n_train"] = data["n_train"]
                                ft_record["n_val"] = data["n_val"]
                                ft_record["n_test"] = data["n_test"]
                                
                                # Adaptar el prefijo test_ar_
                                ft_record["test_mse"] = ft_record.pop("test_ar_mse", None)
                                ft_record["test_rmse"] = ft_record.pop("test_ar_rmse", None)
                                ft_record["test_mae"] = ft_record.pop("test_ar_mae", None)

                                master_results.append(ft_record)
                                pd.DataFrame(master_results).to_csv(out_csv, index=False)
                                logger.info(
                                    Fore.MAGENTA
                                    + f"    [FT-OK] {name}_FT_AR_{mode} ds={ds_idx} seed={seed} → "
                                    + f"test_mse={ft_record.get('test_mse', 'N/A'):.6f}"
                                    + Style.RESET_ALL
                                )
                    # ------ FIN FINE-TUNING RECURSIVO ------

                except Exception as e:
                    logger.error(
                        Fore.RED
                        + f"    [ERROR] {name} ds={ds_idx} seed={seed}: {e}"
                        + Style.RESET_ALL
                    )
                    import traceback
                    traceback.print_exc()
                    continue

                # Liberar memoria GPU
                del model
                torch.cuda.empty_cache()

            # Liberar modelos dict
            del models
            torch.cuda.empty_cache()

    # =================================================================
    # Análisis estadístico final
    # =================================================================
    logger.info(
        Fore.GREEN + "\n" + "=" * 60 + "\n  ANÁLISIS ESTADÍSTICO" + "\n" + "=" * 60
        + Style.RESET_ALL
    )

    df_all = pd.DataFrame(master_results)
    if df_all.empty:
        logger.warning("No hay resultados para analizar.")
        return

    df_all.to_csv(out_csv, index=False)

    # Importar análisis
    from statistical_analysis import (
        generate_summary_table,
        compute_pairwise_comparison,
        generate_full_report,
    )

    test_metrics = ["test_mse", "test_rmse", "test_mae"]
    available_metrics = [m for m in test_metrics if m in df_all.columns]

    if available_metrics:
        report = generate_full_report(
            df_all,
            reference_model="Custom",
            metrics=available_metrics,
        )
        print(report)

        # Guardar reporte
        report_path = os.path.join(exp_dir, "statistical_report.txt")
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report)
        logger.info(f"  Reporte guardado en {report_path}")

        # Tabla resumen
        summary = generate_summary_table(df_all, available_metrics)
        summary_path = os.path.join(exp_dir, "summary_table.csv")
        summary.to_csv(summary_path, index=False)
        logger.info(f"  Tabla resumen guardada en {summary_path}")

        # Comparaciones emparejadas
        comparisons = []
        other_models = [
            m for m in df_all["Modelo"].unique() if m != "Custom"
        ]
        for metric in available_metrics:
            for other in other_models:
                comp = compute_pairwise_comparison(
                    df_all, "Custom", other, metric
                )
                comparisons.append(comp)

        comp_df = pd.DataFrame(comparisons)
        comp_path = os.path.join(exp_dir, "pairwise_comparisons.csv")
        comp_df.to_csv(comp_path, index=False)
        logger.info(f"  Comparaciones guardadas en {comp_path}")

    # Tabla de costo
    cost_cols = ["Modelo", "n_params_trainable", "n_params_total", "train_time_s", "epochs_run"]
    avail_cost = [c for c in cost_cols if c in df_all.columns]
    if len(avail_cost) > 1:
        cost_summary = (
            df_all.groupby("Modelo")[
                [c for c in avail_cost if c != "Modelo"]
            ].mean().round(2)
        )
        cost_path = os.path.join(exp_dir, "cost_summary.csv")
        cost_summary.to_csv(cost_path)
        logger.info(f"  Costos guardados en {cost_path}")
        print("\nCOSTO PROMEDIO POR MODELO:")
        print(cost_summary.to_string())

    logger.info(
        Fore.GREEN
        + "\n==== BENCHMARK FINAL COMPLETADO ===="
        + Style.RESET_ALL
    )

In [5]:
def parse_args():
    p = argparse.Namespace()
    p.base_data_config = "configs/data/real_data1.yaml"
    p.model_config = "configs/model/transformer_base_real_data1.yaml"
    p.training_config = "configs/training/default_real_data1.yaml"
    p.start_dataset = 17
    p.end_dataset = 1
    p.seeds = [42, 84, 126]
    p.skip_ablation = False
    p.skip_baselines = False
    p.reverse_datasets = True
    p.exp_dir = "/content/drive/MyDrive/Memoria/experiments/benchmark_final"
    return p

In [ ]:
main()

[Epoch 001] Step 00050 - train_loss (promedio) = 0.726883
[Epoch 001] Step 00100 - train_loss (promedio) = 0.671768
[Epoch 001] Step 00150 - train_loss (promedio) = 0.661166
[Epoch 001] Step 00200 - train_loss (promedio) = 0.632679
[Epoch 001] Step 00250 - train_loss (promedio) = 0.617044
[Epoch 001] Step 00300 - train_loss (promedio) = 0.604059
[Epoch 001] Step 00350 - train_loss (promedio) = 0.593712
[Epoch 001] Step 00400 - train_loss (promedio) = 0.585348
[Epoch 001] Step 00450 - train_loss (promedio) = 0.579961
[Epoch 001] Step 00500 - train_loss (promedio) = 0.572999
[Epoch 001] Step 00550 - train_loss (promedio) = 0.568360
[Epoch 001] Step 00600 - train_loss (promedio) = 0.563131
[Epoch 001] Step 00650 - train_loss (promedio) = 0.559976
[Epoch 001] Step 00700 - train_loss (promedio) = 0.557107
[Epoch 001] Step 00750 - train_loss (promedio) = 0.553050
[Epoch 001] Step 00800 - train_loss (promedio) = 0.549911
[Epoch 001] Step 00850 - train_loss (promedio) = 0.547577
[Epoch 001] St